In [4]:
import boto3
import os
import logging
from botocore import UNSIGNED
from botocore.client import Config
import pandas as pd
#from rates import *
#from rates2 import *
#from rates3 import *
#from rates5 import *
#from rates6 import *
#import rates6
import os
import pandas as pd
from pathlib import Path

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


In [16]:
# Load metadata
df = pd.read_parquet("CA_baseline_tmy_metadata_and_annual_results.parquet")

# Check column name (might be 'in.utility' or similar)
#print(df.columns[df.columns.str.contains('iou', case=False)])

# Filter for SDGE
sdge_buildings = df[df['in.county_name'] == 'San Diego County'].reset_index()
building_ids = sdge_buildings.index.tolist()


In [ ]:
def download_parquet_files(bucket_name, prefix, local_dir, building_ids=None, max_files=None):
    """
    Download parquet files from an AWS S3 bucket to a local directory.
    
    Args:
        bucket_name (str): The name of the S3 bucket.
        prefix (str): The prefix/folder in the bucket to download files from.
        local_dir (str): The local directory to save downloaded files.
        building_ids (set, optional): Set of building IDs to download. If None, download all.
        max_files (int, optional): Maximum number of files to download. If None, download all files.
    """
    try:
        s3_client = boto3.client('s3', config=Config(signature_version=UNSIGNED))
        
        if not os.path.exists(local_dir):
            os.makedirs(local_dir)
            logger.info(f"Created directory: {local_dir}")
        
        paginator = s3_client.get_paginator('list_objects_v2')
        page_iterator = paginator.paginate(Bucket=bucket_name, Prefix=prefix)
        
        file_count = 0
        for page in page_iterator:
            if 'Contents' not in page:
                logger.warning(f"No objects found with prefix: {prefix}")
                break
                
            for obj in page['Contents']:
                if not obj['Key'].endswith('.parquet'):
                    continue
                
                # Filter by building_ids if provided
                if building_ids is not None:
                    filename = os.path.basename(obj['Key'])
                    bldg_id = int(filename.split('-')[0])
                    if bldg_id not in building_ids:
                        continue
                
                local_file_path = os.path.join(local_dir, os.path.basename(obj['Key']))
                
                logger.info(f"Downloading {obj['Key']} to {local_file_path}...")
                s3_client.download_file(bucket_name, obj['Key'], local_file_path)
                
                file_count += 1
                logger.info(f"Successfully downloaded: {local_file_path}")
                
                if max_files and file_count >= max_files:
                    logger.info(f"Reached maximum number of files to download: {max_files}")
                    return
        
        logger.info(f"Download complete. Downloaded {file_count} files.")
    
    except Exception as e:
        logger.error(f"Error downloading files: {str(e)}")

if __name__ == "__main__":
    # Load metadata and filter for SDGE
    metadata_path = "CA_baseline_tmy_metadata_and_annual_results.parquet"
    df = pd.read_parquet(metadata_path)
    sdge_buildings = df[df['in.county_name'] == 'San Diego County']
    building_ids = set(sdge_buildings.index.tolist())
    
    logger.info(f"Found {len(building_ids)} SDGE buildings")
    
    # Download files
    bucket_name = "oedi-data-lake"
    prefix = "nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_amy2018_release_2/timeseries_individual_buildings/by_state/upgrade=0/state=CA/"
    local_dir = "./Baseline_SDGE"
    
    download_parquet_files(bucket_name, prefix, local_dir, building_ids=building_ids, max_files=None)

In [35]:
import os
import pandas as pd

# Get all SDGE building IDs from metadata
metadata_path = "CA_baseline_tmy_metadata_and_annual_results.parquet"
df = pd.read_parquet(metadata_path)
sdge_buildings = df[df['in.county_name'] == 'San Diego County']
all_sdge_ids = set(sdge_buildings.index.tolist())

# Get already downloaded IDs
baseline_dir = "./Baseline_SDGE"
downloaded_files = os.listdir(baseline_dir)
downloaded_ids = set([int(f.split('-')[0]) for f in downloaded_files if f.endswith('.parquet')])

# Get remaining IDs and take first 2000
remaining_ids = all_sdge_ids - downloaded_ids
new_ids_to_download = set(list(remaining_ids)[:2000])

print(f"Total SDGE buildings: {len(all_sdge_ids)}")
print(f"Already downloaded: {len(downloaded_ids)}")
print(f"Remaining: {len(remaining_ids)}")
print(f"Will download: {len(new_ids_to_download)}")

# Download
# bucket_name = "oedi-data-lake"
# prefix = "nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_amy2018_release_2/timeseries_individual_buildings/by_state/upgrade=0/state=CA/"
# local_dir = "./Baseline_SDGE"

# download_parquet_files(bucket_name, prefix, local_dir, building_ids=new_ids_to_download, max_files=None)


Total SDGE buildings: 4898
Already downloaded: 4317
Remaining: 581
Will download: 581


In [ ]:
import os

# Extract building IDs from already downloaded baseline files
baseline_dir = "./Baseline_SDGE"
downloaded_files = os.listdir(baseline_dir)
building_ids = set([int(f.split('-')[0]) for f in downloaded_files if f.endswith('.parquet')])

print(f"Found {len(building_ids)} buildings in baseline directory")

# Download same buildings for upgrade 11
bucket_name = "oedi-data-lake"
prefix = "nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_amy2018_release_2/timeseries_individual_buildings/by_state/upgrade=11/state=CA/"
local_dir = "./Upgrade11_SDGE"

download_parquet_files(bucket_name, prefix, local_dir, building_ids=building_ids, max_files=None)


Found 2317 buildings in baseline directory


2026-01-29 20:38:08,292 - INFO - Downloading nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_amy2018_release_2/timeseries_individual_buildings/by_state/upgrade=11/state=CA/100190-11.parquet to ./Upgrade11_SDGE/100190-11.parquet...
2026-01-29 20:38:08,669 - INFO - Successfully downloaded: ./Upgrade11_SDGE/100190-11.parquet
2026-01-29 20:38:08,671 - INFO - Downloading nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_amy2018_release_2/timeseries_individual_buildings/by_state/upgrade=11/state=CA/100227-11.parquet to ./Upgrade11_SDGE/100227-11.parquet...
2026-01-29 20:38:09,090 - INFO - Successfully downloaded: ./Upgrade11_SDGE/100227-11.parquet
2026-01-29 20:38:09,092 - INFO - Downloading nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_amy2018_release_2/timeseries_individual_buildings/by_state/upgrade=11/state=CA/100321-11.parquet to ./Upgrade11_SDGE/100321-11.parquet...
2026-01-29 20:38:09

In [26]:
baseline_dir = "./Upgrade11_SDGE"
downloaded_files = os.listdir(baseline_dir)
building_ids = set([int(f.split('-')[0]) for f in downloaded_files if f.endswith('.parquet')])

print(f"Found {len(building_ids)} buildings in baseline directory")


Found 317 buildings in baseline directory


In [20]:
def download_parquet_files(bucket_name, prefix, local_dir, max_files=None):
    """
    Download parquet files from an AWS S3 bucket to a local directory.
    
    Args:
        bucket_name (str): The name of the S3 bucket.
        prefix (str): The prefix/folder in the bucket to download files from.
        local_dir (str): The local directory to save downloaded files.
        max_files (int, optional): Maximum number of files to download. If None, download all files.
    """
    try:
        # Create a session with no credentials (since this is a public bucket)
        s3_client = boto3.client('s3', config=Config(signature_version=UNSIGNED))
        
        # Create local directory if it doesn't exist
        if not os.path.exists(local_dir):
            os.makedirs(local_dir)
            logger.info(f"Created directory: {local_dir}")
        
        # List objects in the bucket with the given prefix
        paginator = s3_client.get_paginator('list_objects_v2')
        page_iterator = paginator.paginate(Bucket=bucket_name, Prefix=prefix)
        
        file_count = 0
        for page in page_iterator:
            if 'Contents' not in page:
                logger.warning(f"No objects found with prefix: {prefix}")
                break
                
            for obj in page['Contents']:
                # Skip if not a parquet file
                if not obj['Key'].endswith('.parquet'):
                    continue
                
                # Create the local file path
                local_file_path = os.path.join(local_dir, os.path.basename(obj['Key']))
                
                # Download the file
                logger.info(f"Downloading {obj['Key']} to {local_file_path}...")
                s3_client.download_file(bucket_name, obj['Key'], local_file_path)
                
                file_count += 1
                logger.info(f"Successfully downloaded: {local_file_path}")
                
                # Check if we've reached the maximum number of files
                if max_files and file_count >= max_files:
                    logger.info(f"Reached maximum number of files to download: {max_files}")
                    return
        
        logger.info(f"Download complete. Downloaded {file_count} files.")
    
    except Exception as e:
        logger.error(f"Error downloading files: {str(e)}")

if __name__ == "__main__":
    # Extract bucket name and prefix from the URL
    bucket_name = "oedi-data-lake"
    prefix = "nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_amy2018_release_2/timeseries_individual_buildings/by_state/upgrade=0/state=CA/"
    #prefix = "nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_amy2018_release_2/timeseries_individual_buildings/by_state/upgrade=11/state=CA/"
    # Set the local directory to save files
    #local_dir = "./Upgrade11"
    local_dir = "./Baseline_SDGE"
    
    # Optional: limit the number of files to download (remove or set to None to download all)
    #max_files = 20
    max_files = 1
    # Download the files
    download_parquet_files(bucket_name, prefix, local_dir, max_files)

2026-01-29 19:02:07,478 - INFO - Downloading nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_amy2018_release_2/timeseries_individual_buildings/by_state/upgrade=11/state=CA/100005-11.parquet to ./Baseline_SDGE/100005-11.parquet...
2026-01-29 19:02:09,454 - INFO - Successfully downloaded: ./Baseline_SDGE/100005-11.parquet
2026-01-29 19:02:09,457 - INFO - Reached maximum number of files to download: 1


### resstock processsing for matto

In [ ]:
#madalsa@ucsb.edu - Google Drive/My Drive/RateDesign/CA_baseline_metadata_and_annual_results.parquet
ca_baseline = pd.read_parquet('CA_baseline_metadata_and_annual_results.parquet').reset_index()

In [ ]:
ca_baseline['weight']

In [ ]:
def get_income_bracket_description(income):
    if income < 20000:
        return "Under $20,000"
    elif 20000 <= income <= 29999:
        return "$20,000 - $29,999"
    elif 30000 <= income <= 39999:
        return "$30,000 - $39,999"
    elif 40000 <= income <= 49999:
        return "$40,000 - $49,999"
    elif 50000 <= income <= 59999:
        return "$50,000 - $59,999"
    elif 60000 <= income <= 69999:
        return "$60,000 - $69,999"
    elif 70000 <= income <= 79999:
        return "$70,000 - $79,999"
    elif 80000 <= income <= 89999:
        return "$80,000 - $89,999"
    elif 90000 <= income <= 99999:
        return "$90,000 - $99,999"
    elif 100000 <= income <= 124999:
        return "$100,000 - $124,999"
    elif 125000 <= income <= 149999:
        return "$125,000 - $149,999"
    elif 150000 <= income <= 174999:
        return "$150,000 - $174,999"
    elif 175000 <= income <= 199999:
        return "$175,000 - $199,999"
    else:  # More than $200,000
        return "More than $200,000"


def get_sqft_bracket_description(sqft):
    if sqft < 500:
        return "Less than 500 square feet"
    elif 500 <= sqft <= 999:
        return "500 - 999 square feet"
    elif 1000 <= sqft <= 1499:
        return "1,000 - 1,499 square feet"
    elif 1500 <= sqft <= 1999:
        return "1,500 - 1,999 square feet"
    elif 2000 <= sqft <= 2499:
        return "2,000 - 2,499 square feet"
    elif 2500 <= sqft <= 2999:
        return "2,500 - 2,999 square feet"
    elif 3000 <= sqft <= 3499:
        return "3,000 - 3,499 square feet"
    elif 3500 <= sqft <= 3999:
        return "3,500 - 3,999 square feet"
    elif 4000 <= sqft <= 4499:
        return "4,000 - 4,499 square feet"
    elif 4500 <= sqft <= 4999:
        return "4,500 - 4,999 square feet"
    elif 5000 <= sqft <= 5499:
        return "5,000 - 5,499 square feet"
    else:  # 5,500 square feet or larger
        return "5,500 square feet or larger"

        
def get_hometype(hometype):
    if hometype == 'Multi-Family with 2 - 4 Units':
        return "Multi-family"

    elif hometype == 'Multi-Family with 5+ Units':
        return "Multi-family"

    elif hometype == 'Single-Family Attached':
        return "Single-family"
        
    elif hometype == 'Single-Family Detached':
        return "Single-family"

    else:
        return "Mobile Home"
        
ca_baseline['income_bracket'] = ca_baseline['in.representative_income'].apply(get_income_bracket_description)
ca_baseline['sqft_bracket'] = ca_baseline['in.sqft'].apply(get_sqft_bracket_description)
ca_baseline['home_type'] = ca_baseline['in.geometry_building_type_recs'].apply(get_hometype)

In [ ]:
len(ca_baseline)

In [ ]:
plt.hist(ca_baseline['in.representative_income'])
plt.title('Histogram of income ResStock Metadata, California')

In [ ]:
crosswalk = ca_baseline.groupby(['in.puma', 'home_type', 'income_bracket', 'sqft_bracket']).count().reset_index()
crosswalk['counts'] = crosswalk['bldg_id']
crosswalk = crosswalk[['in.puma', 'home_type', 'income_bracket', 'sqft_bracket', 'counts']]

In [ ]:
import pandas as pd
import itertools

# Define all possible values for each dimension
puma_values = crosswalk['in.puma'].unique()
home_types = crosswalk['home_type'].unique()
income_types = crosswalk['income_bracket'].unique()
sqft_brackets = crosswalk['sqft_bracket'].unique()

# Create complete multi-index of all combinations
complete_index = pd.MultiIndex.from_product(
    [puma_values, home_types, income_types, sqft_brackets],
    names=['in.puma', 'home_type', 'income_bracket', 'sqft_bracket']
)

# If your data is already grouped/aggregated, set the same multi-index
df_indexed = crosswalk.set_index(['in.puma', 'home_type', 'income_bracket', 'sqft_bracket'])

# Reindex to include all combinations, filling missing with 0
df_complete = df_indexed.reindex(complete_index, fill_value=0)

# Reset index if you want regular columns back
df_complete = df_complete.reset_index()

In [ ]:
puma_totals = df_complete.groupby(['in.puma', 'home_type'])['counts'].sum().reset_index()
puma_totals.columns = ['in.puma', 'home_type', 'total_stock']

In [ ]:
# Merge the total stock back to the original dataframe
df_new = pd.merge(left = df_complete,
                  right = puma_totals,
                  how = 'left',
                  left_on = ['in.puma', 'home_type'],
                  right_on = ['in.puma', 'home_type'])

In [ ]:
df_new['counts'].sum()

In [ ]:
df_new.to_csv('restock_counts_puma_housing.csv')

## running bills

In [ ]:
u = pd.read_excel('retail_rates_data_may202025.xlsx', sheet_name = 'retail_rates_april2025')
u[u['utility'] == 'SMUD']['rate_type'].unique().tolist()

### fixing 2010 PUMA to 2020 puma in resstock

In [ ]:
ca_2income_rescale = pd.read_parquet('CA_Baseline_metadata_rescaled_twoincomes.parquet')
ca_2income_rescale['old_puma'] = ca_2income_rescale['in.puma'].astype(str)

In [ ]:
puma_2020_2010 = pd.read_excel('PUMA2010_PUMA2020_crosswalk.xlsx', sheet_name = 'PUMA2010_PUMA2020')
puma_best = puma_2020_2010.loc[puma_2020_2010.groupby('GISJOIN10')['pPUMA10_Land'].idxmax()]
puma_ca_crosswalk = puma_best[puma_best['State10_Name'] == 'California']
puma_ca_crosswalk['old_puma'] = puma_ca_crosswalk['GISJOIN10'].astype(str)
#puma_2020_2010['old_puma'] = puma_2020_2010['GISJOIN10'].astype(str)

In [ ]:
ca_2income_rescale_puma = pd.merge(ca_2income_rescale,puma_ca_crosswalk, how = 'left', on = 'old_puma')

## running main code

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

# Import your rate calculation functions
# from your_rate_module import calculate_hourly_rates_with_consumption, calculate_total_bill

# Directory containing parquet files
input_dir = "Upgrade11"  # Change this to your directory path

# Building characteristics file
metadata_file = "CA_Baseline_metadata_rescaled_twoincomes_puma20.parquet"  # Your building characteristics CSV

# Baseline PUMA file to map PUMA to IOU
#baseline_puma_file = "baseline_puma.csv"  # Change this to your baseline PUMA file path

# Output directory
output_dir = "bills_upgrade11_june10"
os.makedirs(output_dir, exist_ok=True)

# Load building characteristics metadata
print("Loading building characteristics metadata...")
metadata_df = pd.read_parquet(metadata_file).reset_index()


# Load baseline PUMA file to map PUMA to IOU
print("Loading baseline PUMA to IOU mapping...")
try:
    baseline_puma_df = pd.read_excel('retail_rates_data_may202025.xlsx', sheet_name = 'baseline_puma')
    print(f"Loaded baseline PUMA file with columns: {list(baseline_puma_df.columns)}")
except FileNotFoundError:
    print(f"Warning: {baseline_puma_file} not found. You may need to update the file path.")
    baseline_puma_df = pd.DataFrame()  # Empty dataframe as fallback

# Create a lookup dictionary for building metadata
metadata_lookup = {}
for _, row in metadata_df.iterrows():
    building_id = str(row['building_id'])  # Convert to string to match parquet filenames
    metadata_lookup[building_id] = {
        'income_category': row['income_category'],
        'restock_income': row['restock_income'],  # Add restock income from original data
        'representative_income': row['in.representative_income'],
        'puma': row['puma20'],
        'city': row['in.city'],
        'county': row['in.county'],
        'state': row['in.state'],
        'climate_zone': row['in.cec_climate_zone'],
        'scaling_factor': row['scaling_factor']  # Add scaling factor
    }

print(f"Loaded metadata for {len(metadata_lookup)} buildings")

# Create a lookup dictionary for PUMA to IOU mapping
puma_to_iou = {}
if not baseline_puma_df.empty:
    # Adjust these column names based on your actual baseline_puma file structure
    puma_col = 'puma'  # Change to your PUMA column name
    iou_col = 'iou'    # Change to your IOU column name
    
    # If your file has different column names, update them here
    # Common alternatives might be: 'PUMA', 'utility', 'utility_name', 'IOU', etc.
    if puma_col in baseline_puma_df.columns and iou_col in baseline_puma_df.columns:
        for _, row in baseline_puma_df.iterrows():
            puma = str(row[puma_col])
            iou = row[iou_col]
            puma_to_iou[puma] = iou
        print(f"Created PUMA to IOU mapping for {len(puma_to_iou)} PUMAs")
        print(f"IOUs found: {set(puma_to_iou.values())}")
    else:
        print(f"Warning: Expected columns '{puma_col}' and '{iou_col}' not found in baseline PUMA file")
        print(f"Available columns: {list(baseline_puma_df.columns)}")

# Define rate types by IOU - customize this based on your specific utilities and their rate schedules
iou_rate_types = {
    'PGE': ['E1',
 'E-TOU-C',
'E-ELEC',
 'EV2A',
            
'E-TOU-C-F',
 'EV2A-F',
            
'E-TOU-C-F50',
 'E1-F50',
 'EV2A-F50',
            
'E-TOU-C-F70',
 'E1-F70',
 'EV2A-F70',
                        
 'E1-F100',
 'EV2A-F100',
'E-TOU-C-F100']
    ,
    
'SCE': ['D',
'TOU-D-4-9',
'TOU-D-5-8',
'TOU-PRIME',
        
'TOU-D-4-9-F',
'TOU-D-5-8-F',
        
'D-F50',
'TOU-D-4-9-F50',
'TOU-PRIME-F50',
          
'D-F70',
'TOU-D-4-9-F70',
'TOU-PRIME-F70',
        
'D-F100',
'TOU-D-4-9-F100',
'TOU-PRIME-F100']
,

'SDGE': ['DR',
'TOU-DR1',
'TOU-DR2',
'EV-TOU-5',

'TOU-DR1-F',
'TOU-DR2-F',
            
'DR-F50',
'TOU-DR1-F50',
'EV-TOU-5-F50',
             
 'DR-F70',
 'TOU-DR1-F70',
 'EV-TOU-5-F70',
             
 'DR-F100',
 'TOU-DR1-F100',
 'EV-TOU-5-F100']
,

    'SMUD' : ['R-TOD']
,
    # Add more IOUs and their rate schedules as needed
    'DEFAULT': ['E1', 'E-TOU-C', 'EV2A', 'E-ELEC']  # Fallback rates if IOU not found
}

def get_iou_for_puma(puma):
    """Get the IOU that serves a specific PUMA."""
    return puma_to_iou.get(str(puma), 'DEFAULT')

def get_rate_types_for_iou(iou):
    """Get the available rate types for a specific IOU."""
    return iou_rate_types.get(iou, iou_rate_types['DEFAULT'])

def get_puma_from_metadata(metadata):
    """
    Extract PUMA from metadata for rate calculation.
    The rate calculation function expects PUMA as the location identifier.
    """
    puma = metadata.get('puma')
    if puma and not pd.isna(puma):
        return str(puma)  # Ensure it's a string
    else:
        raise ValueError("PUMA not found in building metadata")

def calculate_bills_for_income_type(puma, available_rate_types, hourly_df, income_level, income_type_suffix):
    """
    Calculate bills for all rate types for a specific income level and return results with suffix.
    """
    bill_results = {}
    
    for rate_type in available_rate_types:
        try:
            # Calculate hourly rates using the building's specific PUMA
            hourly_rates_result = calculate_hourly_rates_with_consumption(
                puma, rate_type, hourly_df
            )
            
            # Calculate bill using the building's specific income level
            bill_result = calculate_total_bill(hourly_rates_result, hourly_df, income_level)
            
            # Store multiple bill components with income type suffix
            bill_results[f"{rate_type}_total_bill{income_type_suffix}"] = bill_result['total_bill']
            bill_results[f"{rate_type}_energy_charges{income_type_suffix}"] = bill_result['energy_charges']['total']
            bill_results[f"{rate_type}_fixed_charges{income_type_suffix}"] = bill_result['fixed_charges']['total']
            bill_results[f"{rate_type}_care_discount{income_type_suffix}"] = bill_result['care_discount']['amount']
            bill_results[f"{rate_type}_average_rate{income_type_suffix}"] = bill_result.get('average_rate', 0)
            
        except Exception as e:
            print(f"    Error calculating {rate_type} for {income_type_suffix}: {str(e)}")
            # Set error values
            bill_results[f"{rate_type}_total_bill{income_type_suffix}"] = None
            bill_results[f"{rate_type}_energy_charges{income_type_suffix}"] = None
            bill_results[f"{rate_type}_fixed_charges{income_type_suffix}"] = None
            bill_results[f"{rate_type}_care_discount{income_type_suffix}"] = None
            bill_results[f"{rate_type}_average_rate{income_type_suffix}"] = None
    
    return bill_results

# Get all parquet files in the directory
parquet_files = list(Path(input_dir).glob("*.parquet"))

# Initialize list to store all results
all_results = []
processed_count = 0
skipped_count = 0

# Process each parquet file
for file_path in parquet_files:
    building_id_raw = file_path.stem
    # Remove "-0" suffix if it exists (for baseline scenarios)
    building_id = building_id_raw.replace("-11", "") if building_id_raw.endswith("-11") else building_id_raw
    print(f"Processing: {building_id_raw} (lookup ID: {building_id})")
    
    # Check if we have metadata for this building
    if building_id not in metadata_lookup:
        print(f"  Warning: No metadata found for building {building_id}, skipping...")
        skipped_count += 1
        continue
    
    # Get building metadata
    building_metadata = metadata_lookup[building_id]
    income_category = building_metadata['income_category']  # Your estimated income
    restock_income = building_metadata['restock_income']    # Original restock income
    scaling_factor = building_metadata['scaling_factor']    # Scaling factor for energy consumption
    
    try:
        puma = get_puma_from_metadata(building_metadata)
    except ValueError as e:
        print(f"  Error: {str(e)}, skipping building {building_id}")
        skipped_count += 1
        continue
    
    # Get IOU for this PUMA
    iou = get_iou_for_puma(puma)
    
    # Get rate types available for this IOU
    available_rate_types = get_rate_types_for_iou(iou)
    
    print(f"  Building metadata - Income Category: {income_category}, Restock Income: {restock_income}, PUMA: {puma}, IOU: {iou}, Scaling Factor: {scaling_factor}")
    print(f"  Available rate types: {available_rate_types}")
    
    try:
        # Read the demand data
        df = pd.read_parquet(file_path)
        timestamp_column = 'timestamp'  # Change this to your actual timestamp column name
        
        if timestamp_column in df.columns:
            # Ensure the timestamp column is parsed as datetime
            df[timestamp_column] = pd.to_datetime(df[timestamp_column])
            df = df.set_index(timestamp_column)

        # Resample to hourly frequency 
        hourly = df.resample('h').mean()
        
        # Get the base hourly consumption and apply scaling factor
        base_hourly_df = hourly['out.electricity.total.energy_consumption'][0:8760]
        scaled_hourly_df = base_hourly_df * scaling_factor
        
        print(f"  Applied scaling factor {scaling_factor}: Original total = {base_hourly_df.sum():.2f} kWh, Scaled total = {scaled_hourly_df.sum():.2f} kWh")

        # Initialize result dictionary for this building
        result_row = {
            'building_id': building_id_raw,
            'income_category': income_category,
            'restock_income': restock_income,
            'puma': puma,
            'iou': iou,
            'climate_zone': building_metadata['climate_zone'],
            'representative_income': building_metadata['representative_income'],
            'city': building_metadata['city'],
            'county': building_metadata['county'],
            'state': building_metadata['state'],
            'scaling_factor': scaling_factor,
            'original_consumption_kwh': base_hourly_df.sum(),
            'scaled_consumption_kwh': scaled_hourly_df.sum()
        }
        
        # Calculate bills using income_category (your estimated income)
        if income_category and not pd.isna(income_category):
            print(f"  Calculating bills with income_category: {income_category}")
            income_cat_results = calculate_bills_for_income_type(
                puma, available_rate_types, scaled_hourly_df, income_category, "_income_cat"
            )
            result_row.update(income_cat_results)
        else:
            print(f"  Warning: income_category is missing or invalid for building {building_id}")
        
        # Calculate bills using restock_income (original data income)
        if restock_income and not pd.isna(restock_income):
            print(f"  Calculating bills with restock_income: {restock_income}")
            restock_results = calculate_bills_for_income_type(
                puma, available_rate_types, scaled_hourly_df, restock_income, "_restock"
            )
            result_row.update(restock_results)
        else:
            print(f"  Warning: restock_income is missing or invalid for building {building_id}")
        
        # Add to results list
        all_results.append(result_row)
        processed_count += 1
        
        print(f"  Completed: {building_id}")
        
    except Exception as e:
        print(f"  Error processing {file_path.name}: {str(e)}")
        skipped_count += 1
        continue

# Convert all results to DataFrame
print(f"\nProcessed {processed_count} buildings successfully")
print(f"Skipped {skipped_count} buildings (missing metadata or errors)")
print(f"Total files processed: {len(parquet_files)}")

if all_results:
    results_df = pd.DataFrame(all_results)
    
    # Save to single CSV file
    output_file = os.path.join(output_dir, "all_buildings_bills_by_puma_iou_income_scaled.csv")
    results_df.to_csv(output_file, index=False)
    
    print(f"\nSaved consolidated results to: {output_file}")
    print(f"CSV contains {len(results_df)} buildings with {len(results_df.columns)} columns")
    
    # Create summary statistics by income category (both versions)
    print("\n=== SUMMARY BY INCOME CATEGORY (Your Estimates) ===")
    income_cat_summary = []
    bill_columns_cat = [col for col in results_df.columns if col.endswith('_total_bill_income_cat')]
    
    for income_cat in results_df['income_category'].unique():
        if pd.isna(income_cat):
            continue
        subset = results_df[results_df['income_category'] == income_cat]
        
        for bill_col in bill_columns_cat:
            rate_type = bill_col.replace('_total_bill_income_cat', '')
            valid_bills = subset[bill_col].dropna()
            
            if len(valid_bills) > 0:
                income_cat_summary.append({
                    'income_category': income_cat,
                    'rate_type': rate_type,
                    'count': len(valid_bills),
                    'mean_bill': valid_bills.mean(),
                    'median_bill': valid_bills.median(),
                    'std_bill': valid_bills.std(),
                    'min_bill': valid_bills.min(),
                    'max_bill': valid_bills.max()
                })
    
    if income_cat_summary:
        income_cat_summary_df = pd.DataFrame(income_cat_summary)
        print(income_cat_summary_df.groupby(['income_category', 'rate_type'])['mean_bill'].first().unstack())
        
        # Save detailed summary for income category
        summary_file_cat = os.path.join(output_dir, "summary_by_income_category_and_rate.csv")
        income_cat_summary_df.to_csv(summary_file_cat, index=False)
        print(f"\nSaved income category summary to: {summary_file_cat}")
    
    print("\n=== SUMMARY BY RESTOCK INCOME (Original Data) ===")
    restock_summary = []
    bill_columns_restock = [col for col in results_df.columns if col.endswith('_total_bill_restock')]
    
    for restock_inc in results_df['restock_income'].unique():
        if pd.isna(restock_inc):
            continue
        subset = results_df[results_df['restock_income'] == restock_inc]
        
        for bill_col in bill_columns_restock:
            rate_type = bill_col.replace('_total_bill_restock', '')
            valid_bills = subset[bill_col].dropna()
            
            if len(valid_bills) > 0:
                restock_summary.append({
                    'restock_income': restock_inc,
                    'rate_type': rate_type,
                    'count': len(valid_bills),
                    'mean_bill': valid_bills.mean(),
                    'median_bill': valid_bills.median(),
                    'std_bill': valid_bills.std(),
                    'min_bill': valid_bills.min(),
                    'max_bill': valid_bills.max()
                })
    
    if restock_summary:
        restock_summary_df = pd.DataFrame(restock_summary)
        print(restock_summary_df.groupby(['restock_income', 'rate_type'])['mean_bill'].first().unstack())
        
        # Save detailed summary for restock income
        summary_file_restock = os.path.join(output_dir, "summary_by_restock_income_and_rate.csv")
        restock_summary_df.to_csv(summary_file_restock, index=False)
        print(f"\nSaved restock income summary to: {summary_file_restock}")
    
    # Create summary by IOU (using income_category version)
    print("\n=== SUMMARY BY IOU ===")
    iou_summary = []
    
    for iou in results_df['iou'].unique():
        if pd.isna(iou):
            continue
        subset = results_df[results_df['iou'] == iou]
        
        for bill_col in bill_columns_cat:  # Using income_category version for IOU summary
            rate_type = bill_col.replace('_total_bill_income_cat', '')
            valid_bills = subset[bill_col].dropna()
            
            if len(valid_bills) > 0:
                iou_summary.append({
                    'iou': iou,
                    'rate_type': rate_type,
                    'count': len(valid_bills),
                    'mean_bill': valid_bills.mean(),
                    'median_bill': valid_bills.median(),
                    'std_bill': valid_bills.std(),
                    'min_bill': valid_bills.min(),
                    'max_bill': valid_bills.max()
                })
    
    iou_summary_df = pd.DataFrame(iou_summary)
    if not iou_summary_df.empty:
        print(iou_summary_df.groupby(['iou', 'rate_type'])['mean_bill'].first().unstack())
        
        iou_file = os.path.join(output_dir, "summary_by_iou_and_rate.csv")
        iou_summary_df.to_csv(iou_file, index=False)
        print(f"\nSaved IOU summary to: {iou_file}")
    
    # Print scaling factor summary
    print("\n=== SCALING FACTOR SUMMARY ===")
    print(f"Scaling factor statistics:")
    print(f"  Mean: {results_df['scaling_factor'].mean():.4f}")
    print(f"  Median: {results_df['scaling_factor'].median():.4f}")
    print(f"  Min: {results_df['scaling_factor'].min():.4f}")
    print(f"  Max: {results_df['scaling_factor'].max():.4f}")
    print(f"  Std: {results_df['scaling_factor'].std():.4f}")
    
    print(f"\nOriginal vs Scaled Consumption:")
    print(f"  Total Original: {results_df['original_consumption_kwh'].sum():,.0f} kWh")
    print(f"  Total Scaled: {results_df['scaled_consumption_kwh'].sum():,.0f} kWh")
    print(f"  Overall scaling ratio: {results_df['scaled_consumption_kwh'].sum() / results_df['original_consumption_kwh'].sum():.4f}")
    
    # Print data quality summary
    print("\n=== DATA QUALITY SUMMARY ===")
    print(f"Buildings with valid income category: {results_df['income_category'].notna().sum()}")
    print(f"Buildings with valid restock income: {results_df['restock_income'].notna().sum()}")
    print(f"Buildings with valid PUMA: {results_df['puma'].notna().sum()}")
    print(f"Buildings with valid IOU: {results_df['iou'].notna().sum()}")
    print(f"Buildings with valid scaling factor: {results_df['scaling_factor'].notna().sum()}")
    
    print(f"\nIncome category distribution:")
    print(results_df['income_category'].value_counts())
    
    print(f"\nRestock income distribution:")
    print(results_df['restock_income'].value_counts())
    
    print(f"\nIOU distribution:")
    print(results_df['iou'].value_counts())
    
else:
    print("No files were processed successfully")

print("Processing complete!")